# 08 — Interactive Member Selection (v2: all xmatch sources)

Like notebook 07, but over **every source in `hst_xmatch/master_combined_v2.csv`**
— including HST-only stars far fainter than Gaia. Selections seed
`bp3m-pop-fit-v2` via `member_seed_v2.csv` (keyed by `source_index` +
`gaia_source_id`).

**How to use** — identical to notebook 07:
1. Run all cells. Box select is default; lasso via the toolbar.
2. Panel selections **intersect**; a re-draw **replaces** a panel's region
   (tick **Extend** to OR); double-click clears a panel. Membership is
   recomputed in Python from the drawn coordinates.
3. Missing photometry is permissive by default (`STRICT_MISSING = False`).
4. **Save member_seed_v2.csv**, run the final summary cell, then:
   `bp3m-pop-fit-v2 --name FIELD --lvd_key KEY --use_member_seed [...]`

**Note on stability**: `source_index` is the row position in
`master_combined_v2.csv`. Re-running `bp3m-v2` regenerates that file and can
reorder rows — redraw the selection after a v2 rerun.

**Setup** — needs `plotly` + `anywidget` in the kernel env; see notebook 07's
header for the JupyterHub labextension symlink fix.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Leo_I'
SEED_OUT   = 'member_seed_v2.csv'  # written into the field directory
VPD_ZOOM   = 3.0                   # initial VPD half-width (mas/yr)
MIN_PAIR   = 50                    # min joint stars for a CMD/CC panel
STRICT_MISSING = False             # True: drawn panel rejects stars lacking data
GAIA_SOURCE = 'v2'                 # 'v2': Gaia-matched stars take v2 posterior PMs
SIG_PM_MAX  = 1.0                  # only show sources with RMS PM sigma below this
N_DET_MIN   = 3                    # ... and at least this many fitted detections
# (SIG_PM_MAX/N_DET_MIN should match the bp3m-pop-fit-v2 eligibility flags:
#  --max_sigma_free_pm / --min_detections — hidden sources can't be selected,
#  and the pop fit would ignore them anyway)
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
from pathlib import Path

field_dir = Path(OUTPUT_DIR).expanduser().resolve() / FIELD_NAME
print('field:', field_dir)

In [ ]:
# ── Load: xmatch master catalogue (+ v2 posterior PMs for Gaia stars) ───────
from bp3m.pipeline.run_pop_fit_v2 import _load_catalog

mc = _load_catalog(field_dir, gaia_source=GAIA_SOURCE)
sig_rms = np.sqrt((mc['sig_ra'] ** 2 + mc['sig_dec'] ** 2) / 2)
n_det_fit = pd.to_numeric(mc.get('n_detect_fit'), errors='coerce').fillna(0)
show = (np.isfinite(mc['pm_ra']) & np.isfinite(mc['pm_dec'])
        & np.isfinite(sig_rms) & (sig_rms > 0) & (sig_rms < SIG_PM_MAX)
        & (n_det_fit >= N_DET_MIN)).to_numpy()
master = mc.loc[show].reset_index(drop=True)
# selection id: source_index as string (plotly customdata must avoid
# JS-number roundtrips; gaia ids especially)
master['sid_str'] = master['source_index'].astype(str)
print(f"{show.sum()} of {len(mc)} sources shown "
      f"(sigma_rms < {SIG_PM_MAX}, n_detect_fit >= {N_DET_MIN}); "
      f"{(master['gaia_source_id'] != 0).sum()} Gaia-matched")

In [ ]:
# ── Panel definitions ────────────────────────────────────────────────────────
_WAVELENGTHS = {  # nm, blue→red ordering
    'F275W': 275, 'F336W': 336, 'F390W': 390, 'F435W': 435, 'F438W': 438,
    'F475W': 475, 'F555W': 555, 'F606W': 606, 'F625W': 625, 'F775W': 775,
    'F814W': 814, 'F850LP': 900, 'F110W': 1100, 'F125W': 1250, 'F160W': 1600,
}

def _wl(band):
    return _WAVELENGTHS.get(band.split('/')[0].upper(), 9999)

mag_cols = sorted((c for c in master.columns if c.startswith('mag_wmean_')),
                  key=lambda c: _wl(c.replace('mag_wmean_', '')))
bands = [c.replace('mag_wmean_', '') for c in mag_cols]
print('bands:', bands)

panels = []   # (title, x, y, xlabel, ylabel, invert_y)
panels.append(('VPD (xmatch/v2)', master['pm_ra'], master['pm_dec'],
               'pmra [mas/yr]', 'pmdec [mas/yr]', False))

def _n_joint(*cols):
    m = np.ones(len(master), bool)
    for c in cols:
        m &= np.isfinite(master[c].to_numpy(float))
    return int(m.sum())

_skipped = []
for i in range(len(bands)):
    for j in range(i + 1, len(bands)):
        b, r = bands[i], bands[j]
        if b.split('/')[0] == r.split('/')[0]:
            _skipped.append(f'{b} × {r} (same filter)')
            continue
        n = _n_joint(f'mag_wmean_{b}', f'mag_wmean_{r}')
        if n < MIN_PAIR:
            _skipped.append(f'{b} × {r} (n={n})')
            continue
        panels.append((f'{b} − {r} CMD  [n={n}]',
                       master[f'mag_wmean_{b}'] - master[f'mag_wmean_{r}'],
                       master[f'mag_wmean_{r}'],
                       f'{b} − {r}', r, True))

if len(bands) >= 3:
    from itertools import combinations
    for b1, b2, b3 in combinations(bands, 3):
        if len({b.split('/')[0] for b in (b1, b2, b3)}) < 3:
            continue
        n = _n_joint(f'mag_wmean_{b1}', f'mag_wmean_{b2}', f'mag_wmean_{b3}')
        if n < MIN_PAIR:
            _skipped.append(f'{b1} × {b2} × {b3} (n={n})')
            continue
        panels.append((f'({b1}−{b2}) vs ({b2}−{b3})  [n={n}]',
                       master[f'mag_wmean_{b1}'] - master[f'mag_wmean_{b2}'],
                       master[f'mag_wmean_{b2}'] - master[f'mag_wmean_{b3}'],
                       f'{b1} − {b2}', f'{b2} − {b3}', False))

# Parallax plane (xmatch fit) vs the reddest available magnitude
if 'parallax_xmatch' in master.columns and bands:
    _red = f'mag_wmean_{bands[-1]}'
    if _n_joint(_red) >= MIN_PAIR:
        panels.append(('Parallax (xmatch)', master[_red],
                       master['parallax_xmatch'],
                       bands[-1], 'parallax [mas]', False))

if _skipped:
    print(f'skipped {len(_skipped)} sparse/degenerate panels:')
    for s in _skipped:
        print('  ', s)
print(f'{len(panels)} panels: ' + ', '.join(p[0] for p in panels))

In [ ]:
# ── Interactive selection UI ─────────────────────────────────────────────────
import plotly.graph_objects as go
import ipywidgets as W

gid_all       = master['sid_str'].to_numpy()
panel_sel     = {}      # panel index -> set of sid_str (None = no constraint)
panel_missing = {}      # panel index -> sources NOT plotted there
figs          = []

status = W.HTML()

def combined_selection():
    active = [(k, s) for k, s in panel_sel.items() if s is not None]
    if not active:
        return None
    out = set(gid_all)
    for k, s in active:
        allowed = s if STRICT_MISSING else (s | panel_missing.get(k, set()))
        out &= allowed
    return out

def _n_partially_constrained(comb):
    active = [k for k, s in panel_sel.items() if s is not None]
    return sum(1 for g in comb
               if any(g in panel_missing.get(k, ()) for k in active))

def refresh():
    comb = combined_selection()
    for fw in figs:
        tr = fw.data[0]
        if comb is None:
            tr.selectedpoints = None
        else:
            ids = tr.customdata[:, 0]
            tr.selectedpoints = [k for k, g in enumerate(ids) if g in comb]
    n = len(comb) if comb is not None else len(gid_all)
    n_active = sum(1 for s in panel_sel.values() if s is not None)
    _extra = ''
    if comb is not None and not STRICT_MISSING:
        n_part = _n_partially_constrained(comb)
        if n_part:
            _extra = (f' — <span style="color:#b8860b">{n_part} selected source'
                      f'{"s" if n_part != 1 else ""} missing data in ≥1 drawn panel</span>')
    status.value = (f'<b>{n}</b> sources selected '
                    f'({n_active} panel constraint{"s" if n_active != 1 else ""} active)'
                    f'{_extra}')

extend_chk = W.Checkbox(value=False, description='Extend (OR new regions into a panel)',
                        indent=False, layout=W.Layout(width='320px'))

def _region_ids(trace, selector):
    # Geometric recompute — never trust plotly's point_inds (see notebook 07).
    xs = np.asarray(trace.x, float)
    ys = np.asarray(trace.y, float)
    if getattr(selector, 'xrange', None) is not None:        # box select
        x0, x1 = sorted(selector.xrange)
        y0, y1 = sorted(selector.yrange)
        mask = (xs >= x0) & (xs <= x1) & (ys >= y0) & (ys <= y1)
    elif getattr(selector, 'xs', None) is not None:          # lasso select
        from matplotlib.path import Path as _MplPath
        poly = _MplPath(np.column_stack([selector.xs, selector.ys]))
        mask = poly.contains_points(np.column_stack([xs, ys]))
    else:
        mask = np.zeros(len(xs), bool)
        mask[list(getattr(selector, 'point_inds', []) or [])] = True
    ids = trace.customdata[:, 0]
    return {ids[k] for k in np.where(mask)[0]}

def _make_handler(panel_idx):
    def _on_select(trace, points, selector):
        ids = _region_ids(trace, selector)
        prev = panel_sel.get(panel_idx)
        if extend_chk.value and prev is not None:
            ids |= prev
        panel_sel[panel_idx] = ids if ids else None
        fw = figs[panel_idx]
        if not extend_chk.value and len(fw.layout.selections or ()) > 1:
            fw.layout.selections = fw.layout.selections[-1:]
        refresh()
    return _on_select

def _make_deselect(panel_idx):
    def _on_deselect(trace, points):
        panel_sel[panel_idx] = None
        figs[panel_idx].layout.selections = ()
        refresh()
    return _on_deselect

for k, (title, x, y, xl, yl, inv) in enumerate(panels):
    fin = np.isfinite(np.asarray(x, float)) & np.isfinite(np.asarray(y, float))
    panel_missing[k] = set(gid_all[~fin])
    fw = go.FigureWidget(
        data=[go.Scattergl(
            x=np.asarray(x, float)[fin], y=np.asarray(y, float)[fin],
            mode='markers',
            marker=dict(size=3, color='#1f77b4'),
            customdata=np.c_[gid_all[fin]],
            selected=dict(marker=dict(color='#d62728', size=5)),
            unselected=dict(marker=dict(opacity=0.15)),
            hovertemplate='%{customdata[0]}<extra></extra>',
        )],
        layout=go.Layout(
            title=dict(text=title, font=dict(size=13)),
            width=430, height=380, dragmode='select',
            margin=dict(l=55, r=10, t=40, b=45),
            xaxis=dict(title=xl), yaxis=dict(title=yl),
        ),
    )
    if inv:
        fw.layout.yaxis.autorange = 'reversed'
    if title.startswith('VPD'):
        fw.layout.xaxis.range = [-VPD_ZOOM, VPD_ZOOM]
        fw.layout.yaxis.range = [-VPD_ZOOM, VPD_ZOOM]
    fw.data[0].on_selection(_make_handler(k))
    fw.data[0].on_deselect(_make_deselect(k))
    figs.append(fw)

def _save(_btn=None):
    comb = combined_selection()
    if comb is None:
        status.value = '<b style="color:red">Nothing selected — draw a box first.</b>'
        return
    sel_mask = master['sid_str'].isin(comb)
    out = master.loc[sel_mask, ['source_index', 'gaia_source_id']].copy()
    out['trusted'] = True
    out = out.sort_values('source_index')
    out_path = field_dir / SEED_OUT
    out.to_csv(out_path, index=False)
    status.value = (f'<b style="color:green">Saved {len(out)} members '
                    f'({int((out.gaia_source_id != 0).sum())} Gaia, '
                    f'{int((out.gaia_source_id == 0).sum())} HST-only) '
                    f'→ {out_path}</b>')

def _clear(_btn=None):
    panel_sel.clear()
    for fw in figs:
        fw.layout.selections = ()
    refresh()

save_btn  = W.Button(description='Save member_seed_v2.csv', button_style='success')
clear_btn = W.Button(description='Clear all selections')
save_btn.on_click(_save)
clear_btn.on_click(_clear)

refresh()
rows = [W.HBox(figs[i:i + 2]) for i in range(0, len(figs), 2)]
W.VBox([W.HBox([save_btn, clear_btn, extend_chk, status]), *rows])

In [ ]:
# ── Final selection summary (run AFTER drawing/saving your selection) ───────
import matplotlib.pyplot as plt

comb = combined_selection()
if comb is None and (field_dir / SEED_OUT).exists():
    _saved = pd.read_csv(field_dir / SEED_OUT)
    comb = set(_saved['source_index'].astype(str))
    print(f'No live selection — loaded {len(comb)} members from {SEED_OUT}')

if comb is None:
    print('No selection drawn and no saved seed CSV — nothing to summarise.')
else:
    sel_mask = master['sid_str'].isin(comb).to_numpy()
    ncol = 3
    nrow = int(np.ceil(len(panels) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.0 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (title, x, y, xl, yl, inv) in zip(axes, panels):
        xv = np.asarray(x, float); yv = np.asarray(y, float)
        fin = np.isfinite(xv) & np.isfinite(yv)
        ax.scatter(xv[fin & ~sel_mask], yv[fin & ~sel_mask],
                   s=2, c='0.8', lw=0, label='not selected')
        ax.scatter(xv[fin & sel_mask], yv[fin & sel_mask],
                   s=5, c='crimson', lw=0, label='selected member')
        ax.set_xlabel(xl); ax.set_ylabel(yl)
        ax.set_title(f'{title}  ({int((fin & sel_mask).sum())} sel)', fontsize=10)
        if inv:
            ax.invert_yaxis()
        if title.startswith('VPD'):
            ax.set_xlim(-VPD_ZOOM, VPD_ZOOM); ax.set_ylim(-VPD_ZOOM, VPD_ZOOM)
    for ax in axes[len(panels):]:
        ax.set_visible(False)
    axes[0].legend(fontsize=8, loc='upper right')
    fig.suptitle(f'{FIELD_NAME} — v2 member selection: '
                 f'{int(sel_mask.sum())} of {len(master)} shown sources', y=1.001)
    fig.tight_layout()
    out_png = field_dir / 'member_seed_v2_selection.png'
    fig.savefig(out_png, dpi=150, bbox_inches='tight')
    print(f'Saved summary figure → {out_png}')
    plt.show()

## Using the selection

```bash
# seed only: starting member set; the fit refines membership freely
bp3m-pop-fit-v2 --name FIELD --lvd_key KEY --use_member_seed [...]

# frozen: the fit may REMOVE seed sources but can never add outsiders
bp3m-pop-fit-v2 --name FIELD --lvd_key KEY --use_member_seed --freeze_member_seed [...]
```

Both auto-load `FIELD/member_seed_v2.csv` (fallback `member_seed.csv`);
`--member_seed_csv PATH` overrides. The seed matches on `source_index`
(row of `master_combined_v2.csv`) plus `gaia_source_id` where present.